# 04 — Image Analysis

Validação das features extraídas por CLIP e YOLOv8:
- Distribuição dos scores semânticos CLIP
- Correlação dos scores com preço
- Exemplos de imagens por score
- Objetos detectados pelo YOLO e impacto no preço

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

PROCESSED_PATH = Path('../data/processed')
IMAGES_PATH = Path('../data/images')

## 1. CLIP Scores — Distribuição e Correlação

In [ ]:
clip_path = PROCESSED_PATH / 'clip_features.parquet'
final_path = PROCESSED_PATH / 'final_features.parquet'

if not clip_path.exists():
    print('CLIP features não encontradas. Rode o pipeline: make up → trigger feature_engineering_dag')
else:
    clip = pd.read_parquet(clip_path)
    final = pd.read_parquet(final_path)
    
    # Merge com preço
    if 'id' in final.columns:
        df = clip.merge(final[['id', 'log_price']], left_on='listing_id', right_on='id', how='inner')
    else:
        df = clip.copy()
    
    print(f'Listings com imagem: {len(clip)}')
    clip_score_cols = ['luxury_score', 'cleanliness_score', 'brightness_score',
                       'professional_photo_score', 'modern_style_score']
    print(clip[[c for c in clip_score_cols if c in clip.columns]].describe())

In [ ]:
if clip_path.exists():
    score_cols = [c for c in clip_score_cols if c in df.columns]
    
    fig, axes = plt.subplots(1, len(score_cols), figsize=(16, 4))
    for ax, col in zip(axes, score_cols):
        df[col].hist(bins=30, ax=ax, color='steelblue', edgecolor='white')
        ax.set_title(col.replace('_score', '').replace('_', ' ').title())
        ax.set_xlabel('Score CLIP (0-1)')
    
    plt.suptitle('Distribuição dos Scores Semânticos CLIP', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
if clip_path.exists() and 'log_price' in df.columns:
    # Correlação com preço
    corr = df[score_cols + ['log_price']].corr()['log_price'].drop('log_price').sort_values()
    
    fig, ax = plt.subplots(figsize=(7, 3))
    colors = ['green' if v > 0 else 'red' for v in corr]
    corr.plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title('Correlação CLIP Scores vs log(Preço)')
    ax.set_xlabel('Pearson r')
    plt.tight_layout()
    plt.show()
    print(corr.to_string())

## 2. Exemplos Visuais por Score

In [ ]:
def show_images_by_score(df, score_col, n=4, title=''):
    """Mostra as N imagens com maior e menor score."""
    if not IMAGES_PATH.exists() or score_col not in df.columns:
        print(f'Imagens não disponíveis para {score_col}')
        return
    
    top = df.nlargest(n, score_col)['listing_id'].tolist()
    bot = df.nsmallest(n, score_col)['listing_id'].tolist()
    
    fig, axes = plt.subplots(2, n, figsize=(4*n, 5))
    fig.suptitle(f'{title or score_col} — Top (acima) vs Bottom (abaixo)', fontsize=12)
    
    for i, listing_id in enumerate(top):
        img_path = IMAGES_PATH / f'{listing_id}.jpg'
        if img_path.exists():
            axes[0, i].imshow(mpimg.imread(str(img_path)))
            score = df.loc[df['listing_id'] == listing_id, score_col].values[0]
            axes[0, i].set_title(f'{score:.2f}', fontsize=9)
        axes[0, i].axis('off')
    
    for i, listing_id in enumerate(bot):
        img_path = IMAGES_PATH / f'{listing_id}.jpg'
        if img_path.exists():
            axes[1, i].imshow(mpimg.imread(str(img_path)))
            score = df.loc[df['listing_id'] == listing_id, score_col].values[0]
            axes[1, i].set_title(f'{score:.2f}', fontsize=9)
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

if clip_path.exists():
    show_images_by_score(clip, 'luxury_score', n=4, title='Luxury Score')
    show_images_by_score(clip, 'brightness_score', n=4, title='Brightness Score')

## 3. YOLO — Objetos Detectados

In [ ]:
yolo_path = PROCESSED_PATH / 'yolo_features.parquet'

if not yolo_path.exists():
    print('YOLO features não encontradas.')
else:
    yolo = pd.read_parquet(yolo_path)
    
    has_cols = [c for c in yolo.columns if c.startswith('has_')]
    detection_rate = yolo[has_cols].mean().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    detection_rate.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Taxa de Detecção de Objetos (YOLO)')
    ax.set_xlabel('Fração de imagens com o objeto')
    ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.show()

In [ ]:
if yolo_path.exists() and final_path.exists():
    final = pd.read_parquet(final_path)
    
    if 'id' in final.columns:
        yolo_merged = yolo.merge(final[['id', 'log_price']], left_on='listing_id', right_on='id', how='inner')
    else:
        yolo_merged = yolo.copy()
    
    if 'log_price' in yolo_merged.columns:
        # Impacto dos objetos no preço
        impacts = []
        for col in has_cols:
            has = yolo_merged[col] == 1
            if has.sum() > 20:
                price_with = np.expm1(yolo_merged.loc[has, 'log_price']).median()
                price_without = np.expm1(yolo_merged.loc[~has, 'log_price']).median()
                impacts.append({
                    'objeto': col.replace('has_', ''),
                    'preco_com': price_with,
                    'preco_sem': price_without,
                    'premium_pct': (price_with / price_without - 1) * 100,
                    'n': has.sum(),
                })
        
        impact_df = pd.DataFrame(impacts).sort_values('premium_pct', ascending=False)
        
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['green' if v > 0 else 'red' for v in impact_df['premium_pct']]
        ax.barh(impact_df['objeto'], impact_df['premium_pct'], color=colors)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title('Premium de Preço por Objeto Detectado (YOLO)')
        ax.set_xlabel('% diferença de preço mediano')
        plt.tight_layout()
        plt.show()
        print(impact_df.to_string(index=False))

## 4. Comparação: Features de Imagem vs Features Tabulares

In [ ]:
# Testar impacto das imagens no modelo — treinar com e sem
import xgboost as xgb
from sklearn.model_selection import KFold
from src.training.train import _load_features, _rmse_original_space, XGBOOST_PARAMS

X, y = _load_features()
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Sem features de imagem
image_cols = [c for c in X.columns if c.startswith(('clip_', 'yolo_', 'has_', 'object_', 'bed_count', 'luxury_score'))]
X_no_img = X.drop(columns=image_cols, errors='ignore')

params = {k: v for k, v in XGBOOST_PARAMS.items() if k != 'early_stopping_rounds'}

oof_no_img = np.zeros(len(y))
oof_with_img = np.zeros(len(y))

for train_idx, val_idx in kf.split(X):
    # Sem imagem
    m1 = xgb.XGBRegressor(**params)
    m1.fit(X_no_img.iloc[train_idx], y.iloc[train_idx])
    oof_no_img[val_idx] = m1.predict(X_no_img.iloc[val_idx])
    
    # Com imagem
    m2 = xgb.XGBRegressor(**params)
    m2.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_with_img[val_idx] = m2.predict(X.iloc[val_idx])

rmse_no_img = _rmse_original_space(y.values, oof_no_img)
rmse_with_img = _rmse_original_space(y.values, oof_with_img)

print(f'RMSE sem imagens:  R${rmse_no_img:.2f}')
print(f'RMSE com imagens:  R${rmse_with_img:.2f}')
print(f'Ganho das imagens: R${rmse_no_img - rmse_with_img:.2f} ({(rmse_no_img - rmse_with_img)/rmse_no_img*100:.1f}%)')